# CVD_특징 기여 — 어떤 입력 특징이 답변에 도움이 되는가?

입력·실행·핵심 결과 순서로 읽습니다. 기본 화면은 요약표이며 `SHOW_DETAILS=True`일 때만 세부 표를 표시합니다. 기존 모델은 변경하지 않습니다.
특징 기여는 비교용 임시 헤드를 학습합니다. 추가 복원 특징 재학습 비교는 별도 선택입니다.

PECVD 원본 Test를 사용하는 내부 연구입니다. 외부 제공 주입 CSV를 읽는 시험은 notebooks/CVD_predict_external_csv.ipynb입니다.


In [ ]:
from pathlib import Path
import sys, importlib, os
os.environ['HF_HUB_OFFLINE']='1'
os.environ['TRANSFORMERS_OFFLINE']='1'
ROOT=next(p for p in (Path.cwd().resolve(),*Path.cwd().resolve().parents) if (p/'src'/'pecvd').is_dir())
if str(ROOT/'src') not in sys.path: sys.path.insert(0,str(ROOT/'src'))
import pecvd.experiments.pipeline_settings as pipeline_settings
pipeline_settings=importlib.reload(pipeline_settings)
PIPELINE_SETTINGS=pipeline_settings.configure_pipeline(ROOT)
if PIPELINE_SETTINGS.get('decision_system')!='fusion_answers':
    raise RuntimeError('These experiments require the trained fusion_answers model.')
from pecvd.experiments.questions import PRIMARY_GOAL_VERSION
from pecvd.experiments.fusion_answers import SYSTEM as GEVIS_SYSTEM
print({'model':GEVIS_SYSTEM,'training_profile':PIPELINE_SETTINGS['training_profile'],'weights':'existing STEP1/2/3'})


## 1. 학습된 Fast GEVIS 로딩과 실험 설정

STEP 3가 완료됐다면 각 EXP의 커널을 재시작하고 처음부터 실행합니다. 원본 CSV를 읽거나 가중치를 다운로드하지 않습니다.
결과는 기밀이며 자동 파일 저장·업로드는 하지 않습니다.


In [ ]:
from dataclasses import replace
import pandas as pd
import pecvd.experiments.routing_audit as runtime_refresh
runtime_refresh=importlib.reload(runtime_refresh)
runtime_refresh.refresh_evaluation_modules()
for _module in ('fusion_answers','fusion_metrics','fast_experiments'):
    importlib.reload(importlib.import_module('pecvd.experiments.'+_module))
from pecvd.experiments import fusion_answers as fusion_answers, fast_experiments as fast
RUNTIME=fast.load_runtime(ROOT)
SEED=42
SELECTED_STEPS=None
MAX_WINDOWS_PER_STEP=16
CONFIG=replace(fusion_answers.FusionConfig(),seed=SEED,max_windows_per_step=MAX_WINDOWS_PER_STEP)
print({'decision':'question-conditioned Fusion + trained selected-center head',
       'question_scope':'fixed English questions; sensor selection is separate',
       'answer_head_parameters':sum(p.numel() for p in RUNTIME.module.FUSION_ANSWER_HEAD.parameters()),
       'deployed_weights_updated':False})
CONTROL_EPOCHS=30
CONTROL_FIT_SECONDS=60.
CONTROL_VIEWS=fast.FEATURE_VIEWS
CONTROL_CONFIG=replace(CONFIG,epochs=CONTROL_EPOCHS,max_seconds=CONTROL_FIT_SECONDS)

SHOW_DETAILS=False
RUN_RECONSTRUCTION_CONTROLS=False


## 2. 실제 Fast 모델 실험 실행

모델이 없거나 설정이 맞지 않으면 중단합니다. 데이터나 모델을 모의 결과로 대체하지 않습니다.


In [ ]:
REPORT=fast.run_feature_controls(RUNTIME,steps=SELECTED_STEPS,config=CONTROL_CONFIG,views=CONTROL_VIEWS)
SUMMARY=REPORT.summary;DETAILS=REPORT.details
display(SUMMARY[[c for c in ('variant','answer_type','questions','answer_accuracy','macro_f1') if c in SUMMARY]])


## 학습 조건과 클래스별 비교

`selected_numeric_only`가 전체 특징 대조군과 비슷하면 높은 점수를 Fusion의 효과로 주장할 수 없습니다.
`position_only`가 높으면 데이터 생성 순서·위치와 정답의 연관도 의심해야 합니다.
한 seed·정해진 합성 과제의 차이이며 실제 고장 성능이나 통계적 유의성을 입증하지 않습니다.


In [ ]:
if SHOW_DETAILS:
    for _name,_report in REPORT.reports.items():
        print(_name)
        display(_report.diagnostics['per_class'])
    ERROR_CASES=DETAILS.loc[~DETAILS.correct].copy()
    display(ERROR_CASES[['variant','step','answer_type','center_sensor','label','prediction']].head(20))


## 복원 오차 수치 특징의 기여

TRAIN/Validation/Test에 같은 사례·정답을 사용해 전체 입력과 명시적 복원 통계 제거를
비교합니다. STEP1·STEP2와 기본 모델은 고정하고 임시 답변 헤드만 학습합니다.
MSE와 복원 예측 차이 통계만 제거하며, 복원 학습을 거친 latent 내용까지 제거하는
실험은 아닙니다. 새로운 CHOICE 종류로 복원 오차를 취급하지 않습니다.


In [ ]:
if globals().get('RUN_RECONSTRUCTION_CONTROLS',False):
    import pecvd.experiments.reconstruction_evidence as reconstruction_evidence
    reconstruction_evidence=importlib.reload(reconstruction_evidence)
    RECON_CONTROLS=reconstruction_evidence.run_controls(RUNTIME,config=CONTROL_CONFIG,steps=SELECTED_STEPS)
    display(RECON_CONTROLS.summary)
    display(RECON_CONTROLS.diagnostics['training'])
